In [1]:
!pip install -q ultralytics

from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/ap-10k.zip"

import zipfile
import os

EXTRACT_PATH = "/content/ap10k"

os.makedirs(EXTRACT_PATH, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted!")

import os

for root, dirs, files in os.walk(EXTRACT_PATH):
    print(root)
    if len(files) > 0:
        print(" Example file:", files[0])
    print("-" * 50)
    break



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset extracted!
/content/ap10k
--------------------------------------------------


In [2]:
import json
import shutil
from pathlib import Path
from PIL import Image

ANNOTATION_FILE = f"{EXTRACT_PATH}/ap-10k/annotations/ap10k-val-split1.json"
IMAGE_DIR = f"{EXTRACT_PATH}/ap-10k/data"

In [3]:
YOLO_DATASET = "/content/ap10k_yolo"

images_out = f"{YOLO_DATASET}/images/train"
labels_out = f"{YOLO_DATASET}/labels/train"

os.makedirs(images_out, exist_ok=True)
os.makedirs(labels_out, exist_ok=True)

In [4]:
with open(ANNOTATION_FILE, 'r') as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]

image_lookup = {img["id"]: img for img in images}

In [5]:
import os

ROOT = "/content/ap10k"

for root, dirs, files in os.walk(ROOT):
    print("\nFOLDER:", root)

    if len(files) > 0:
        print("FILES:", files[:5])

    if "json" in str(files).lower():
        print("JSON FILES FOUND!")


FOLDER: /content/ap10k

FOLDER: /content/ap10k/ap-10k

FOLDER: /content/ap10k/ap-10k/data
FILES: ['000000020324.jpg', '000000032084.jpg', '000000020333.jpg', '000000037835.jpg', '000000037640.jpg']

FOLDER: /content/ap10k/ap-10k/annotations
FILES: ['ap10k-train-split2.json', 'ap10k-train-split3.json', 'ap10k-test-split1.json', 'ap10k-val-split2.json', 'ap10k-train-split1.json']
JSON FILES FOUND!


In [6]:
selected_annotations = annotations[:1000]

used_images = set()

for ann in selected_annotations:

    image_id = ann["image_id"]

    if image_id in used_images:
        continue

    used_images.add(image_id)

    img_info = image_lookup[image_id]

    file_name = img_info["file_name"]
    width = img_info["width"]
    height = img_info["height"]

    img_path = os.path.join(IMAGE_DIR, file_name)

    if not os.path.exists(img_path):
        continue

    shutil.copy(img_path, images_out)

    x, y, w, h = ann["bbox"]

    x_center = (x + w / 2) / width
    y_center = (y + h / 2) / height
    w_norm = w / width
    h_norm = h / height

    keypoints = ann["keypoints"]

    yolo_kpts = []

    for i in range(0, len(keypoints), 3):
        kx = keypoints[i] / width
        ky = keypoints[i + 1] / height
        v = keypoints[i + 2]

        yolo_kpts.extend([kx, ky, v])

    # class_id = 0 (single class animal)
    line = f"0 {x_center} {y_center} {w_norm} {h_norm} " + \
           " ".join(map(str, yolo_kpts))

    label_name = Path(file_name).stem + ".txt"

    with open(os.path.join(labels_out, label_name), "w") as f:
        f.write(line)

print(f"Prepared {len(used_images)} images!")

Prepared 776 images!


In [7]:
yaml_text = """
path: /content/ap10k_yolo

train: images/train
val: images/train

kpt_shape: [17, 3]

names:
  0: animal
"""

with open("/content/ap10k.yaml", "w") as f:
    f.write(yaml_text)

print("YAML file created!")

YAML file created!


In [8]:
from ultralytics import YOLO

# Load pretrained pose model
model = YOLO("yolov8n-pose.pt")

model.train(
    data="/content/ap10k.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="ap10k_pose",
    name="yolov8n_pose"
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ap10k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_pose-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100

ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ec9ee26f2c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034, 

In [9]:
metrics = model.val()

print(metrics)

results = model.predict(
    source="/content/ap10k_yolo/images/train",
    save=True,
    conf=0.25
)

print("Inference complete!")

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-pose summary (fused): 82 layers, 3,289,964 parameters, 0 gradients, 9.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2618.8±1254.5 MB/s, size: 259.1 KB)
val: Scanning /content/ap10k_yolo/labels/train.cache... 776 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 776/776 325.5Mit/s 0.0s
train: /content/ap10k_yolo/images/train/000000037868.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 3.5it/s 13.9s
                   all        776        776      0.945      0.961      0.987      0.849      0.697      0.652      0.517      0.144
Speed: 1.6ms preprocess, 6.0ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/pose/val
ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
b

In [10]:
import os

for root, dirs, files in os.walk("/content"):
    if "best.pt" in files:
        print("FOUND:", os.path.join(root, "best.pt"))

FOUND: /content/runs/pose/ap10k_pose/yolov8n_pose-2/weights/best.pt


In [12]:
import shutil

BEST_MODEL = "/content/runs/pose/ap10k_pose/yolov8n_pose-2/weights/best.pt"

DRIVE_SAVE = "/content/drive/MyDrive/best_ap10k_pose.pt"

shutil.copy(BEST_MODEL, DRIVE_SAVE)

print("Model saved to Google Drive!")

Model saved to Google Drive!


In [1]:
%%writefile custom_pose_loss.py
import torch
import torch.nn as nn

from ultralytics.utils.loss import v8PoseLoss


class CustomPoseLoss(v8PoseLoss):

    def __init__(self, model):
        super().__init__(model)

        self.skeleton_links = [
            (5, 7),
            (7, 9),
            (6, 8),
            (8, 10),
            (11, 13),
            (12, 14)
        ]

        self.target_lengths = torch.tensor([
            0.12,
            0.10,
            0.12,
            0.10,
            0.15,
            0.15
        ])

        self.geom_weight = 0.001

    def geometric_loss(self, pred_kpts):
        device = pred_kpts.device

        targets = self.target_lengths.to(device)

        geom_loss = 0.0

        for i, (j1, j2) in enumerate(self.skeleton_links):

            p1 = pred_kpts[:, j1, :2]
            p2 = pred_kpts[:, j2, :2]

            actual_len = torch.norm(p1 - p2, dim=-1)

            geom_loss += ((actual_len - targets[i]) ** 2).mean()

        return geom_loss

    def __call__(self, preds, batch):
        total_loss, loss_items = super().__call__(preds, batch)

        pred_distri, pred_scores, pred_kpts = preds

        B, A, C = pred_kpts.shape

        num_kpts = self.kpt_shape[0]

        # reshape keypoints
        pred_kpts = pred_kpts.view(
            B,
            A,
            num_kpts,
            3
        )

        conf = pred_scores.sigmoid().max(-1)[0]

        conf_mask = conf > 0.7

        if conf_mask.sum() > 0:

            selected_kpts = pred_kpts[conf_mask]

            geom_loss = self.geometric_loss(selected_kpts)

            if not hasattr(self, "iteration"):
                self.iteration = 0

            self.iteration += 1

            warmup_iters = 1000

            warmup_factor = min(
                1.0,
                self.iteration / warmup_iters
            )

            current_geom_weight = (
                self.geom_weight * warmup_factor
            )

            total_loss += geom_loss * current_geom_weight

        else:
          geom_loss = torch.tensor(
              0.0,
              device=total_loss.device
          )

        loss_items = torch.cat([
            loss_items,
            geom_loss.unsqueeze(0)
        ])

        return total_loss, loss_items

Overwriting custom_pose_loss.py


In [2]:
from ultralytics import YOLO
from ultralytics.models.yolo.pose.train import PoseTrainer

from custom_pose_loss import CustomPoseLoss

In [3]:
class CustomTrainer(PoseTrainer):

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(
            cfg,
            weights,
            verbose
        )

        return model

    def get_validator(self):
        validator = super().get_validator()

        self.loss = CustomPoseLoss(self.model)

        return validator

    def criterion(self, preds, batch):
        return self.loss(preds, batch)

In [4]:
model = YOLO("/content/runs/pose/ap10k_pose/yolov8n_pose-2/weights/best.pt")

In [5]:
model.train(
    data="/content/ap10k.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    trainer=CustomTrainer
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ap10k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/pose/ap10k_pose/yolov8n_pose-2/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer

ultralytics.utils.metrics.PoseMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7865404323f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(P)', 'F1-Confidence(P)', 'Precision-Confidence(P)', 'Recall-Confidence(P)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034, 